# ASL: Preserve Missing-Frame Positions vs Drop Them

Use your Kaggle remote Python kernel. This notebook is self-contained.

**Hypothesis:** removing every frame with no visible hand discards where tracking gaps occur and compresses time. Test that change alone with two 126-feature body-reference inputs: `drop_missing_frames` (the existing notebook 03 baseline) and `preserve_timeline` (keep no-hand rows as zero coordinates and zero validity masks before resampling).

Both variants use identical body reference estimation, model architecture, parameter count, label mapping, signer partitions, training seed, optimizer, batch size, and epoch budget. Neither variant interpolates or reconstructs missing coordinates. Only the frame grid supplied to uniform resampling changes.

Default cost: **four full-data fits, up to 20 epochs each**, using split seeds 43/44 and training seed 42. Inspect your Kaggle compute budget before running the training cell. For a pipeline check use `SAMPLE_LIMIT = 1000` and `EPOCHS = 2`.

Here timeline means row positions in the supplied sequence, including rows with no hands. It does not restore frames missing from the parquet itself or recover original video timestamps. For clips longer than 32 frames, full-grid resampling can omit visible-hand observations that the drop-grid variant would retain. Improvement is not guaranteed.


In [ ]:
print('Initializing missing-frame timeline experiment...', flush=True)
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import shutil
import json
import math
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('Importing TensorFlow...', flush=True)
import tensorflow as tf
print('TensorFlow imported.', flush=True)
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

SPLIT_SEEDS = [43, 44]
TRAIN_SEED = 42
SUBSET_SEED = 42
SAMPLE_LIMIT = None
EPOCHS = 20
N_FRAMES = 32
BATCH_SIZE = 64
VALIDATION_FRACTION = 0.2
VARIANTS = {'drop_missing_frames': 126, 'preserve_timeline': 126}
DATA_DIR = None
INPUT_DIR = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/asl-project')
FOCUS_SIGNS = ['mouth', 'lips', 'chin', 'nose', 'tongue']

if not INPUT_DIR.is_dir():
    raise RuntimeError('Select your Kaggle remote kernel; /kaggle/input is unavailable.')
if len(SPLIT_SEEDS) < 2 or len(set(SPLIT_SEEDS)) != len(SPLIT_SEEDS):
    raise ValueError('Provide at least two distinct split seeds.')
if any(not isinstance(seed, int) or seed < 0 for seed in SPLIT_SEEDS + [TRAIN_SEED, SUBSET_SEED]):
    raise ValueError('All seeds must be nonnegative integers.')
if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < 2:
    raise ValueError('SAMPLE_LIMIT must be None or at least 2.')
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))
print('Scheduled fits:', len(SPLIT_SEEDS) * len(VARIANTS))
print('Mode:', 'FULL DATASET' if SAMPLE_LIMIT is None else 'PIPELINE CHECK')


In [ ]:
# To specify the dataset manually, use Path('/kaggle/input/competitions/asl-signs').
# Search only a few directory levels to avoid traversing every parquet file.
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            'ASL competition data was not found. In your Kaggle notebook, use Add Input to add '
            'Google - Isolated Sign Language Recognition (asl-signs), then rerun this cell.'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple dataset directories found; set DATA_DIR explicitly above: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv is missing required columns: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv contains no samples')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('Label indices must be contiguous and start at zero')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv contains labels missing from the official label mapping')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())


In [ ]:
if table['participant_id'].nunique() < 3:
    raise ValueError('At least three signers are required for this split comparison.')
if table['path'].duplicated().any():
    raise ValueError('Sample paths must be unique.')
if SAMPLE_LIMIT is not None:
    # One shared subset for the entire study, before signer splitting.
    table = table.sample(n=min(SAMPLE_LIMIT, len(table)), random_state=SUBSET_SEED)
table = table.reset_index(drop=True)
if table['participant_id'].nunique() < 3:
    raise ValueError('The selected subset must include at least three signers.')
N_CLASSES = len(labels)
class_names = [dict((v, k) for k, v in labels.items())[i] for i in range(N_CLASSES)]

def split_hash(rows):
    text = rows[['path', 'sign', 'participant_id']].to_csv(index=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

ROOT_DIR = OUTPUT_ROOT / ('timeline_ablation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
ROOT_DIR.mkdir(parents=True, exist_ok=False)
root_config = dict(split_seeds=SPLIT_SEEDS, training_seed=TRAIN_SEED,
    subset_seed=SUBSET_SEED, sample_limit=SAMPLE_LIMIT, epochs=EPOCHS,
    frames=N_FRAMES, batch_size=BATCH_SIZE, validation_fraction=VALIDATION_FRACTION,
    variants=VARIANTS, n_classes=N_CLASSES, tensorflow_version=tf.__version__,
    metadata_hash=split_hash(table), labels=labels,
    feature_version='body_reference_missing_frame_grid_v1')
(ROOT_DIR/'study_config.json').write_text(json.dumps(root_config, indent=2))
(ROOT_DIR/'labels.json').write_text(json.dumps(labels, indent=2))
table.to_csv(ROOT_DIR/'cache_sample_order.csv', index=False)
print('Shared cache samples:', len(table))
print('Study directory:', ROOT_DIR)


## One paired cache; identical normalization

Each raw clip is loaded once. A 252-column disk-backed cache stores the 126 drop-grid features followed by the 126 full-grid features. Both models receive **126**, not 252, input features. Partition views select the appropriate half and the same sample indices.

Body center and scale are calculated from frames that have at least one visible hand, matching notebook 03 for both variants. All-missing-hand clips are zeros for both. The full-grid variant retains absent-hand rows as zeros with zero masks; the drop-grid variant removes those rows first. Both then uniformly select/repeat 32 frame positions. No interpolation, augmentation, or model change is introduced.

Per-clip diagnostics record missing-frame fraction and how many distinct visible source rows survive each sampling strategy. Reports compare validation accuracy within missingness strata; those descriptive associations are not causal estimates. Overlapping validation cohorts remain development data, not an independent final test.


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def clip_reference(frames, hands, valid):
    """Return a stable origin, positive scalar scale, and reference source."""
    xy = np.asarray(frames[..., :2], dtype=np.float64)
    shoulders = xy[:, [489 + 11, 489 + 12], :]
    shoulder_valid = np.isfinite(shoulders).all(axis=(1, 2))
    pairs = shoulders[shoulder_valid]
    if len(pairs):
        widths = np.linalg.norm(pairs[:, 0] - pairs[:, 1], axis=1)
        usable = np.isfinite(widths) & (widths > 1e-6)
        if usable.any():
            center = np.median(pairs[usable].mean(axis=1), axis=0)
            scale = float(np.median(widths[usable]))
            return center, max(scale, 1e-6), 'shoulders'

    face = xy[:, :468, :]
    centers, spreads = [], []
    for points in face:
        points = points[np.isfinite(points).all(axis=1)]
        if len(points) < 3:
            continue
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        if np.isfinite(spread) and spread > 1e-6:
            centers.append(center)
            spreads.append(spread)
    if centers:
        return np.median(centers, axis=0), float(np.median(spreads)), 'face'

    points = np.asarray(hands[valid], dtype=np.float64)
    if len(points):
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        return center, max(spread, 1e-6), 'hands_fallback'
    return np.zeros(2), 1.0, 'no_hands'


def preprocess_drop(frames, length, return_reference=False):
    """Normalize hand xy by a clip-level body reference; keep 126 features."""
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    hands, valid = hands[useful], valid[useful]
    if not len(hands):
        result = np.zeros((length, 126), dtype=np.float32)
        return (result, 'no_hands') if return_reference else result
    center, scale, source = clip_reference(frames[useful], hands, valid)
    safe = np.where(valid[..., None], hands, center)
    normalized = np.where(valid[..., None], (safe - center) / scale, 0)
    features = np.concatenate([normalized.reshape(len(hands), 84), valid], axis=1)
    indices = np.rint(np.linspace(0, len(features) - 1, length)).astype(int)
    result = features[indices].astype(np.float32)
    return (result, source) if return_reference else result


def preprocess_pair(frames, length):
    dropped, reference = preprocess_drop(frames, length, return_reference=True)
    hands = frames[:, list(range(468,489))+list(range(522,543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    full_indices = np.rint(np.linspace(0,len(frames)-1,length)).astype(int)
    if useful.any():
        center, scale, checked_reference = clip_reference(frames[useful],hands[useful],valid[useful])
        assert checked_reference == reference
        safe = np.where(valid[...,None],hands,center)
        normalized = np.where(valid[...,None],(safe-center)/scale,0)
        full = np.concatenate([normalized.reshape(len(frames),84),valid],axis=1)[full_indices].astype(np.float32)
        dropped_indices = np.rint(np.linspace(0,int(useful.sum())-1,length)).astype(int)
        drop_unique = len(np.unique(dropped_indices))
        full_visible_unique = len(np.unique(full_indices[useful[full_indices]]))
    else:
        full = np.zeros((length,126),dtype=np.float32)
        drop_unique = full_visible_unique = 0
    diagnostics = dict(raw_frames=len(frames),visible_hand_frames=int(useful.sum()),
        no_hand_frame_fraction=float((~useful).mean()),
        drop_visible_rows_retained=int(drop_unique),
        full_visible_rows_retained=int(full_visible_unique),reference_source=reference)
    return np.concatenate([dropped,full],axis=1), diagnostics


In [ ]:
estimated_bytes = len(table) * N_FRAMES * 252 * 4
free_bytes = shutil.disk_usage(ROOT_DIR).free
print(f'Feature cache: {estimated_bytes / 1024**3:.2f} GiB; free disk: {free_bytes / 1024**3:.2f} GiB')
if free_bytes < estimated_bytes + 512 * 1024**2:
    raise RuntimeError('Insufficient disk space for the shared cache plus output headroom.')
feature_path = ROOT_DIR/'paired_timeline_features.npy'
cache = np.lib.format.open_memmap(feature_path, mode='w+', dtype=np.float32,
                                 shape=(len(table), N_FRAMES, 252))
reference_counts = {}
quality_records = []
for index, row in enumerate(table.itertuples(index=False)):
    features, diagnostics = preprocess_pair(load_frames(DATA_DIR/row.path), N_FRAMES)
    reference = diagnostics['reference_source']
    quality_records.append(dict(path=row.path, **diagnostics))
    if features.shape != (N_FRAMES, 252) or not np.isfinite(features).all():
        raise ValueError(f'Invalid features for {row.path}')
    cache[index] = features
    reference_counts[reference] = reference_counts.get(reference, 0) + 1
    if (index+1) % 500 == 0 or index+1 == len(table):
        print(f'Preprocessed {index+1}/{len(table)}', flush=True)
cache.flush()
del cache
x_all = np.load(feature_path, mmap_mode='r')
y_all = table['sign'].map(labels).to_numpy(dtype=np.int64)
(ROOT_DIR/'reference_counts.json').write_text(json.dumps(reference_counts, indent=2))
quality_table = pd.DataFrame(quality_records)
quality_table['visibility_group'] = pd.cut(quality_table['no_hand_frame_fraction'],
    bins=[-0.000001,0,0.25,0.5,1], labels=['none_missing','up_to_25pct','25_to_50pct','over_50pct'])
quality_table.to_csv(ROOT_DIR/'sampling_diagnostics.csv',index=False)

class PartitionFeatures:
    def __init__(self, cache, indices, offset=0):
        self.cache = cache
        self.offset = offset
        self.indices = np.asarray(indices, dtype=np.int64)
    def __getitem__(self, key):
        rows, times, features = key
        shifted = slice(self.offset+(features.start or 0),self.offset+features.stop,features.step)
        return self.cache[self.indices[rows],times,shifted]
    def for_variant(self, variant):
        if variant not in VARIANTS:
            raise ValueError('Unknown timeline variant')
        return PartitionFeatures(self.cache,self.indices,126 if variant=='preserve_timeline' else 0)


def activate_split(split_seed):
    global SPLIT_SEED, STUDY_DIR, STUDY_SIGNATURE, train_rows, val_rows
    global x_train, y_train, x_val, y_val
    SPLIT_SEED = split_seed
    splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION,
                                random_state=split_seed)
    train_indices, val_indices = next(splitter.split(table, groups=table['participant_id']))
    train_rows = table.iloc[train_indices].reset_index(drop=True)
    val_rows = table.iloc[val_indices].reset_index(drop=True)
    train_signers, val_signers = set(train_rows['participant_id']), set(val_rows['participant_id'])
    assert train_signers.isdisjoint(val_signers)
    x_train, y_train = PartitionFeatures(x_all, train_indices), y_all[train_indices]
    x_val, y_val = PartitionFeatures(x_all, val_indices), y_all[val_indices]
    STUDY_DIR = ROOT_DIR/f'split_{split_seed}'
    STUDY_DIR.mkdir(exist_ok=True)
    config = dict(root_config, active_split_seed=split_seed,
        train_split_hash=split_hash(train_rows), validation_split_hash=split_hash(val_rows),
        train_signers=[str(x) for x in sorted(train_signers)],
        validation_signers=[str(x) for x in sorted(val_signers)])
    STUDY_SIGNATURE = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    config_path = STUDY_DIR/'split_config.json'
    if config_path.is_file() and json.loads(config_path.read_text()) != config:
        raise ValueError('Existing split configuration does not match this study.')
    config_path.write_text(json.dumps(config, indent=2))
    train_rows.to_csv(STUDY_DIR/'train_split.csv', index=False)
    val_rows.to_csv(STUDY_DIR/'validation_split.csv', index=False)
    np.savez(STUDY_DIR/'cache_indices.npz', train_indices=train_indices, validation_indices=val_indices)
    print('Split:', split_seed, '| train:', len(train_rows), '| validation:', len(val_rows))
    print('Held-out signers:', sorted(val_signers))


In [ ]:
def make_dataset(x, y, n_features, training_seed, shuffle):
    rng = np.random.default_rng(training_seed)
    def batches():
        indices = rng.permutation(len(y)) if shuffle else np.arange(len(y))
        for start in range(0, len(y), BATCH_SIZE):
            batch = indices[start:start+BATCH_SIZE]
            yield np.asarray(x[batch, :, :n_features], dtype=np.float32), y[batch]
    ds = tf.data.Dataset.from_generator(batches, output_signature=(
        tf.TensorSpec((None, N_FRAMES, n_features), tf.float32),
        tf.TensorSpec((None,), tf.int64)))
    return ds.apply(tf.data.experimental.assert_cardinality(math.ceil(len(y)/BATCH_SIZE))).prefetch(1)

def build_model(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input((N_FRAMES, n_features)),
        tf.keras.layers.Conv1D(96, 5, padding='same', activation='relu'),
        tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(N_CLASSES, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def fit_one(variant, training_seed):
    n_features = VARIANTS[variant]
    run_dir = STUDY_DIR / f'{variant}_seed_{training_seed}'
    run_dir.mkdir(exist_ok=True)
    # metrics.json is written last and acts as a completion marker.
    metrics_path = run_dir / 'metrics.json'
    required = ['best.keras', 'history.json', 'classification_report.csv', 'validation_predictions.csv']
    if metrics_path.is_file():
        saved = json.loads(metrics_path.read_text())
        if (saved.get('study_signature') != STUDY_SIGNATURE or
            saved.get('training_seed') != training_seed or saved.get('variant') != variant):
            raise ValueError('Completed run metadata does not match this study.')
        if not all((run_dir/name).is_file() for name in required):
            raise ValueError('Completed run is missing required artifacts.')
        print('Reusing completed run:', run_dir.name)
        return saved
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(training_seed)
    train_ds = make_dataset(x_train.for_variant(variant), y_train, n_features, training_seed, True)
    val_ds = make_dataset(x_val.for_variant(variant), y_val, n_features, training_seed, False)
    model = build_model(n_features)
    run_config = dict(study_signature=STUDY_SIGNATURE, variant=variant, split_seed=SPLIT_SEED,
                      training_seed=training_seed, n_features=n_features,
                      model_parameters=int(model.count_params()))
    (run_dir/'config.json').write_text(json.dumps(run_config, indent=2))
    print('Training:', variant, '| seed:', training_seed, '| parameters:', model.count_params())
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, shuffle=False,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(run_dir/'best.keras'),
                monitor='val_accuracy', mode='max', save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=5,
                restore_best_weights=True),
            tf.keras.callbacks.CSVLogger(str(run_dir/'training_log.csv')),
        ])
    history_data = {k: [float(v) for v in values] for k, values in history.history.items()}
    (run_dir/'history.json').write_text(json.dumps(history_data, indent=2))
    best = tf.keras.models.load_model(run_dir/'best.keras')
    scores = {k: float(v) for k, v in best.evaluate(val_ds, return_dict=True).items()}
    def features_only(features, targets):
        return features
    probabilities = best.predict(val_ds.map(features_only))
    predicted = probabilities.argmax(axis=1)
    assert len(predicted) == len(val_rows)
    report = classification_report(y_val, predicted, labels=list(range(N_CLASSES)),
        target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(run_dir/'classification_report.csv')
    result = val_rows[['path', 'participant_id', 'sign']].copy()
    result['predicted_sign'] = [class_names[int(i)] for i in predicted]
    result['confidence'] = probabilities.max(axis=1)
    result.to_csv(run_dir/'validation_predictions.csv', index=False)
    metrics = dict(run_config, **scores, run_dir=str(run_dir),
        best_epoch=int(np.argmax(history_data['val_accuracy'])+1),
        epochs_run=len(history_data['accuracy']), pipeline_check=SAMPLE_LIMIT is not None,
        validation_samples=len(y_val))
    metrics_path.write_text(json.dumps(metrics, indent=2))
    return metrics


## Train the paired models for each split

This cell starts four full-data fits by default. A completed run can be reused when this training cell is rerun within the same study; an unfinished run restarts from epoch one. Rerunning the root configuration creates a new study and does not resume an older one. Kernel/session recovery and partial-epoch checkpoints are not implemented.


In [ ]:
records = []
for split_seed in SPLIT_SEEDS:
    activate_split(split_seed)
    for variant in VARIANTS:
        records.append(fit_one(variant, TRAIN_SEED))
        pd.DataFrame(records).to_csv(ROOT_DIR/'run_metrics.csv', index=False)
print('Completed fits:', len(records))


In [ ]:
runs = pd.DataFrame(records)
expected = {(variant, seed) for variant in VARIANTS for seed in SPLIT_SEEDS}
if set(zip(runs['variant'], runs['split_seed'])) != expected or len(runs) != len(expected):
    raise ValueError('Every split requires exactly one completed run of both variants.')
paired = runs.pivot(index='split_seed', columns='variant', values='accuracy')
paired['accuracy_gain_pp'] = 100*(paired['preserve_timeline']-paired['drop_missing_frames'])
paired.to_csv(ROOT_DIR/'paired_split_accuracy.csv')
summary = runs.groupby('variant').agg(splits=('accuracy','size'),
    mean_accuracy=('accuracy','mean'), std_accuracy=('accuracy','std'),
    min_accuracy=('accuracy','min'), max_accuracy=('accuracy','max'), mean_loss=('loss','mean'))
summary.to_csv(ROOT_DIR/'split_summary.csv')
display(summary)
display(paired)
print(f"Descriptive mean paired gain: {paired['accuracy_gain_pp'].mean():+.2f} percentage points")
print('Preserved-timeline wins:', int((paired['accuracy_gain_pp']>0).sum()), '/', len(paired))
print('These validation cohorts can overlap; standard deviations are descriptive, not confidence intervals.')
if SAMPLE_LIMIT is not None:
    print('PIPELINE CHECK ONLY: do not report these as full-task results.')
fig, ax = plt.subplots(figsize=(8,4))
for variant in VARIANTS:
    ax.plot(paired.index.astype(str), paired[variant], marker='o', label=variant)
ax.set(xlabel='Signer split seed', ylabel='Validation accuracy', title=f'Training seed fixed at {TRAIN_SEED}')
ax.legend()
fig.tight_layout()
fig.savefig(ROOT_DIR/'signer_split_accuracy.png', dpi=150)
plt.show()


In [ ]:
focus_records, changed_frames, cohorts = [], [], {}
for split_seed in SPLIT_SEEDS:
    directory = ROOT_DIR/f'split_{split_seed}'
    validation_rows = pd.read_csv(directory/'validation_split.csv')
    cohorts[split_seed] = set(validation_rows['participant_id'].astype(str))
    frames = {}
    for variant in VARIANTS:
        row = runs[(runs['split_seed']==split_seed) & (runs['variant']==variant)].iloc[0]
        run_dir = Path(row['run_dir'])
        predictions = pd.read_csv(run_dir/'validation_predictions.csv')
        columns = ['path','participant_id','sign']
        if not predictions[columns].astype(str).equals(validation_rows[columns].astype(str)):
            raise ValueError('Predictions are not aligned with the current split.')
        frames[variant] = predictions
        report = pd.read_csv(run_dir/'classification_report.csv', index_col=0)
        for sign in FOCUS_SIGNS:
            if sign in labels:
                focus_records.append(dict(split_seed=split_seed, variant=variant, sign=sign,
                    recall=float(report.loc[sign,'recall']), support=int(report.loc[sign,'support'])))
    body, combined = frames['drop_missing_frames'], frames['preserve_timeline']
    changed = body[['path','participant_id','sign']].copy()
    changed['split_seed'] = split_seed
    changed['dropped_prediction'] = body['predicted_sign']
    changed['preserved_prediction'] = combined['predicted_sign']
    changed['dropped_correct'] = body['predicted_sign'].eq(body['sign'])
    changed['preserved_correct'] = combined['predicted_sign'].eq(combined['sign'])
    changed_frames.append(changed[changed['dropped_prediction']!=changed['preserved_prediction']])
focus_table = pd.DataFrame(focus_records, columns=['split_seed','variant','sign','recall','support'])
focus_table.to_csv(ROOT_DIR/'focus_class_recall_by_split.csv',index=False)
display(focus_table)
changes = pd.concat(changed_frames, ignore_index=True)
changes.to_csv(ROOT_DIR/'prediction_disagreements_by_split.csv',index=False)
changes[changes['sign']=='tongue'].to_csv(ROOT_DIR/'tongue_disagreements_by_split.csv',index=False)
overlap = pd.DataFrame([[len(cohorts[a]&cohorts[b]) for b in SPLIT_SEEDS] for a in SPLIT_SEEDS],
    index=SPLIT_SEEDS, columns=SPLIT_SEEDS)
overlap.to_csv(ROOT_DIR/'validation_signer_overlap.csv')
print('Held-out signer overlap (diagonal is cohort size):')
display(overlap)
print('Outputs:', ROOT_DIR)


In [ ]:
# Compare both variants on the same missingness strata, within each split.
strata = []
for run in runs.to_dict('records'):
    predictions = pd.read_csv(Path(run['run_dir'])/'validation_predictions.csv')
    joined = predictions.merge(quality_table,on='path',how='left',validate='one_to_one')
    if joined['visibility_group'].isna().any():
        raise ValueError('Missing sampling diagnostics for validation rows.')
    joined['correct'] = joined['sign'].eq(joined['predicted_sign'])
    for group,rows in joined.groupby('visibility_group',observed=True):
        strata.append(dict(split_seed=run['split_seed'],variant=run['variant'],
            visibility_group=str(group),support=len(rows),accuracy=float(rows['correct'].mean()),
            mean_drop_visible_rows_retained=float(rows['drop_visible_rows_retained'].mean()),
            mean_full_visible_rows_retained=float(rows['full_visible_rows_retained'].mean())))
strata_table = pd.DataFrame(strata)
strata_table.to_csv(ROOT_DIR/'accuracy_by_missingness.csv',index=False)
display(strata_table)
strata_paired = strata_table.pivot(index=['split_seed','visibility_group'],columns='variant',values='accuracy')
strata_paired['accuracy_gain_pp'] = 100*(strata_paired['preserve_timeline']-strata_paired['drop_missing_frames'])
strata_paired.to_csv(ROOT_DIR/'paired_accuracy_by_missingness.csv')
display(strata_paired)
print('These are validation strata, not a reason to remove difficult examples.')


## Interpret the timing ablation

First compare paired best-checkpoint accuracy per signer split and inspect the original missingness strata. Both models have identical parameter counts; only the time grid differs. The full-grid variant may help retain gap location, but can spend its finite frame budget on absent hands and omit visible observations in long clips. Inspect sampling diagnostics alongside accuracy.

If both splits do not show a useful consistent gain, retain the original drop-grid baseline. Do not filter validation samples or infer wrong labels from these selected errors. Two overlapping splits with one training seed are exploratory evidence, not a final test.

Save the study/split configurations, labels, sample partitions, checkpoints, metrics, sampling diagnostics, missingness reports, and plots before the Kaggle session ends. No original video timing is recovered and no TFLite submission is generated.
